###load_struct_to_prep_customers
####Builds the Gold layer customers dimension table (ecom_prep.customers) - implements SCD Type 2 logic with surrogate keys, start_date, end_date, and is_current flag

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from datetime import date

In [0]:
struct_customers = spark.table("ecom_struct.customers")
geo = spark.table("ecom_ref_struct.geolocation")

# join customers with geolocation
customers = struct_customers.join(
    geo,
    struct_customers.customer_zip_code_prefix == geo.geolocation_zip_code_prefix,
    "left"
).select(
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_city").alias("city"),
    col("customer_state").alias("state"),
    col("customer_zip_code_prefix").alias("zip_code_prefix"),
    col("geolocation_lat"),
    col("geolocation_lng")
)
customers.createOrReplaceTempView("customers")

In [0]:
window_cust = Window.partitionBy("customer_unique_id").orderBy(col("customer_id").desc())

unique_customers = (customers.withColumn("rn", row_number().over(window_cust))
                       .filter(col("rn") == 1)
                       .drop("rn"))
unique_customers.createOrReplaceTempView("unique_customers")

In [0]:
%sql

select count(*) from customers

In [0]:
%sql

select count(*) from unique_customers

In [0]:
old_customers = spark.table("ecom_prep.customers").filter(col("is_current") == True)

customer_compared = unique_customers.join(
    old_customers.select(
        col("customer_unique_id").alias("g_customer_unique_id"),
        col("city").alias("g_city"),
        col("state").alias("g_state"),
        col("zip_code_prefix").alias("g_zip_code_prefix")
    ),
    unique_customers.customer_unique_id == col("g_customer_unique_id"),
    "left"
)

tagged_customers = customer_compared.withColumn(
    "action",
    when(col("g_customer_unique_id").isNull(), lit("NEW"))
    .when(
        (col("city") != col("g_city")) |
        (col("state") != col("g_state")) |
        (col("zip_code_prefix") != col("g_zip_code_prefix")),
        lit("CHANGED")
    )
    .otherwise(lit("UNCHANGED"))
)


# for merge — only changed records
tagged_customers.filter(col("action") == "CHANGED").createOrReplaceTempView("changed_records")

# for insert — new + changed records
tagged_customers.filter(col("action").isin(["NEW", "CHANGED"])).createOrReplaceTempView("new_records")

In [0]:
%sql

select count(*) from changed_records

In [0]:
%sql

select count(*) from new_records

In [0]:
spark.sql("""
    MERGE INTO ecom_prep.customers AS tgt
    USING changed_records AS src
    ON tgt.customer_unique_id = src.customer_unique_id
    AND tgt.is_current = true
    WHEN MATCHED THEN UPDATE SET
        tgt.is_current = false,
        tgt.end_date = date_sub(current_date(), 1)
""").display()

In [0]:
spark.sql('''
    INSERT INTO ecom_prep.customers (
        customer_unique_id,
        city,
        state,
        zip_code_prefix,
        geolocation_lat,
        geolocation_lng,
        is_current,
        start_date,
        end_date
    )
    SELECT
        customer_unique_id,
        city,
        state,
        zip_code_prefix,
        geolocation_lat,
        geolocation_lng,
        true AS is_current,
        current_date() AS start_date,
        CAST('9999-12-31' AS DATE) AS end_date
    FROM new_records
''').display()

####Customer Mapping

In [0]:
# all customer_id to customer_unique_id mappings
window_map = Window.partitionBy("customer_id").orderBy("customer_id")

mapped_customers = customers.withColumn("rn", row_number().over(window_map)) \
                         .filter(col("rn") == 1) \
                         .drop("rn") \
                         .select(
                             col("customer_id"),
                             col("customer_unique_id")
                         )

mapped_customers.createOrReplaceTempView("mapped_customers")

In [0]:
spark.sql('''
    MERGE INTO ecom_prep.customers_map AS tgt
    USING mapped_customers AS src
    ON tgt.customer_unique_id = src.customer_unique_id
    WHEN MATCHED THEN
        UPDATE SET
            tgt.customer_id = src.customer_id
    WHEN NOT MATCHED THEN
        INSERT (
            customer_id,
            customer_unique_id
        )
        VALUES (
            src.customer_id,
            src.customer_unique_id
        )
''').display()